In [1]:
!nvidia-smi

Thu Apr 30 09:22:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/VuTrinhNguyenHoang/bidirectional-table-text-alignment.git

Cloning into 'bidirectional-table-text-alignment'...
remote: Enumerating objects: 192, done.
remote: Counting objects: 100% (192/192), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 192 (delta 90), reused 153 (delta 54), pack-reused 0 (from 0)
Receiving objects: 100% (192/192), 56.52 KiB | 7.06 MiB/s, done.
Resolving deltas: 100% (90/90), done.


In [4]:
%cd bidirectional-table-text-alignment

/kaggle/working/bidirectional-table-text-alignment


In [4]:
!mkdir -p outputs/checkpoints outputs/data/processed
!cp -a /kaggle/input/models/trnhvu2/bitt/pytorch/default/1/checkpoints/. outputs/checkpoints/
!cp -a /kaggle/input/models/trnhvu2/bitt/pytorch/default/1/data/processed/. outputs/data/processed/

In [5]:
!pip -q install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.3 MB/s eta 0:00:00


In [6]:
from pathlib import Path
import json

MODE = "small"  # "small", "medium", "debug", "full"
GENERATOR_MODE = "full" 
SELECTOR_MODE = "medium"
CONFIG = "configs/main.yaml"
TOP_K = 3
THRESHOLD_STEP = 0.01

In [7]:
tuning_path = Path(f"outputs/metrics/{MODE}/cell_selector_threshold_tuning_valid_top{TOP_K}.json")
tuning = json.loads(tuning_path.read_text(encoding="utf-8"))
BEST_THRESHOLD = tuning["best_threshold"]
BEST_THRESHOLD, tuning["best_metrics"]

(0.88,
 {'cell_precision': 0.8251666666666666,
  'cell_recall': 0.7365003252102165,
  'cell_f1': 0.747639296953809,
  'cell_exact_match': 0.326,
  'threshold': 0.88,
  'top_k': 3,
  'fallback_rate': 0.521,
  'empty_threshold_rate': 0.003,
  'overflow_threshold_rate': 0.518,
  'avg_selected_count': 2.627,
  'candidate_filter_rate': 0.032,
  'avg_table_cells': 236.605,
  'avg_candidate_cells': 159.597})

In [8]:
!PYTHONPATH=. python3 scripts/evaluation_e2e.py --mode {MODE} --generator-mode {GENERATOR_MODE} --selector-mode {SELECTOR_MODE} --config {CONFIG} --top_k {TOP_K} --threshold {BEST_THRESHOLD}

Loading weights: 100%|█| 131/131 [00:00<00:00, 1757.19it/s, Materializing param=
Loading weights: 100%|█| 104/104 [00:00<00:00, 1859.92it/s, Materializing param=
config_sentence_transformers.json: 100%|████████| 116/116 [00:00<00:00, 653kB/s]
README.md: 10.5kB [00:00, 25.7MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:00<00:00, 289kB/s]
config.json: 100%|█████████████████████████████| 612/612 [00:00<00:00, 3.57MB/s]
model.safetensors: 100%|███████████████████| 90.9M/90.9M [00:00<00:00, 93.6MB/s]
Loading weights: 100%|█| 103/103 [00:00<00:00, 1918.02it/s, Materializing param=
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_config.json: 100%|███████████████████| 350/350 [00:00<00:00, 2.88MB/s]

In [9]:
threshold_tag = f"threshold{BEST_THRESHOLD:.4f}".replace(".", "p")
metric_path = Path(f"outputs/metrics/{MODE}/e2e_metrics_{threshold_tag}_top{TOP_K}.json")
metrics = json.loads(metric_path.read_text(encoding="utf-8"))
metrics

{'mode': 'small',
 'generator_mode': 'full',
 'selector_mode': 'medium',
 'top_k': 3,
 'threshold': 0.88,
 'selection_strategy': 'threshold_with_top_k_fallback',
 'cell_selection': {'cell_precision': 0.8251666666666666,
  'cell_recall': 0.7365003252102165,
  'cell_f1': 0.747639296953809,
  'cell_exact_match': 0.326},
 'consistency': {'lexical': {'bleu': {'score': 28.804976731471463,
    'counts': [9695, 5504, 3535, 2357],
    'totals': [14286, 13286, 12286, 11286],
    'precisions': [67.86364272714546,
     41.427066084600334,
     28.772586684030603,
     20.8842814105972],
    'bp': 0.7989830785785365,
    'sys_len': 14286,
    'ref_len': 17492},
   'rouge': {'rouge1': 0.6328060045749534,
    'rouge2': 0.41182090567664936,
    'rougeL': 0.5475936315607821,
    'rougeLsum': 0.5469717863421264}},
  'semantic': {'cosine_mean': 0.817167681708932,
   'cosine_std': 0.13997710398612673,
   'cosine_min': 0.057302042841911316,
   'cosine_max': 1.0000001192092896},
  'number_faithfulness': {'n